In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install -q tensorflow keras opencv-python matplotlib pandas scikit-learn

In [ ]:
!pip install -q albumentations

In [ ]:
import os

os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)


In [ ]:
import numpy as np
import tensorflow as tf
import random

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
!free -h   # RAM usage
!df -h     # Disk usage

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.5Gi       8.0Gi       2.0Mi       3.2Gi        10Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
tmpfs           6.4G  484K  6.4G   1% /var/colab
/dev/sda1       114G   22G   93G  19% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git /content/data/PlantDoc

Cloning into '/content/data/PlantDoc'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 41.34 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.


In [ ]:
# ================================================================
# DIAGNOSTIC — Run this FIRST (takes ~2 min)
# Tells us exactly why accuracy is stuck at 54%
# ================================================================

import os
import numpy as np
import tensorflow as tf
from PIL import Image
from pathlib import Path
from collections import Counter

TRAIN_DIR = "/content/data/PlantDoc/train"
TEST_DIR  = "/content/data/PlantDoc/test"

print("=" * 55)
print("DIAGNOSTIC REPORT")
print("=" * 55)

# ── 1. GPU check ─────────────────────────────────────────────
print("\n[1] GPU:")
gpus = tf.config.list_physical_devices("GPU")
print(f"    Found: {[g.name for g in gpus]}")
print(f"    TF version: {tf.__version__}")

# ── 2. Class counts ──────────────────────────────────────────
print("\n[2] Class sample counts (train):")
train_cls = sorted([d for d in os.listdir(TRAIN_DIR)
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))])
test_cls  = sorted([d for d in os.listdir(TEST_DIR)
                    if os.path.isdir(os.path.join(TEST_DIR, d))])

train_counts, test_counts = {}, {}
for cls in train_cls:
    files = [f for f in os.listdir(os.path.join(TRAIN_DIR, cls))
             if f.lower().endswith((".jpg",".jpeg",".png"))]
    train_counts[cls] = len(files)

for cls in test_cls:
    files = [f for f in os.listdir(os.path.join(TEST_DIR, cls))
             if f.lower().endswith((".jpg",".jpeg",".png"))]
    test_counts[cls] = len(files)

total_train = sum(train_counts.values())
total_test  = sum(test_counts.values())
min_cls     = min(train_counts, key=train_counts.get)
max_cls     = max(train_counts, key=train_counts.get)
classes_lt50= [c for c,n in train_counts.items() if n < 50]

print(f"    Total train images : {total_train}")
print(f"    Total test images  : {total_test}")
print(f"    Train classes      : {len(train_cls)}")
print(f"    Test  classes      : {len(test_cls)}")
print(f"    Min class (train)  : '{min_cls}' = {train_counts[min_cls]} images")
print(f"    Max class (train)  : '{max_cls}' = {train_counts[max_cls]} images")
print(f"    Classes with <50 train images: {len(classes_lt50)}")
for c in sorted(classes_lt50):
    print(f"      • {c}: {train_counts[c]}")

# ── 3. Class mismatch ────────────────────────────────────────
print("\n[3] Class alignment:")
only_train = set(train_cls) - set(test_cls)
only_test  = set(test_cls)  - set(train_cls)
common     = set(train_cls) & set(test_cls)
print(f"    Common classes     : {len(common)}")
print(f"    Only in train      : {only_train}")
print(f"    Only in test       : {only_test}")

# ── 4. Corrupted file scan ───────────────────────────────────
print("\n[4] Scanning for corrupted files …")
bad = []
for root in [TRAIN_DIR, TEST_DIR]:
    for path in Path(root).rglob("*"):
        if path.suffix.lower() not in {".jpg",".jpeg",".png"}: continue
        try:
            img = Image.open(path); img.verify()
            img = Image.open(path); img.load()
            arr = np.array(img.convert("RGB")).astype(np.float32)
            if np.isnan(arr).any() or np.isinf(arr).any():
                raise ValueError("NaN pixels")
        except Exception as e:
            bad.append((str(path), str(e)))

if bad:
    print(f"    ⚠ Found {len(bad)} corrupted files:")
    for p, e in bad[:10]:
        print(f"      • {os.path.basename(p)}: {e}")
else:
    print(f"    ✓ No corrupted files found.")

# ── 5. Image size distribution ───────────────────────────────
print("\n[5] Image size sample (first 50 train images):")
sizes = []
for cls in list(train_counts.keys())[:5]:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for f in list(os.listdir(cls_dir))[:10]:
        if f.lower().endswith((".jpg",".jpeg",".png")):
            try:
                img = Image.open(os.path.join(cls_dir, f))
                sizes.append(img.size)
            except: pass

if sizes:
    ws = [s[0] for s in sizes]; hs = [s[1] for s in sizes]
    print(f"    Width  range : {min(ws)} – {max(ws)} px")
    print(f"    Height range : {min(hs)} – {max(hs)} px")
    print(f"    Avg size     : {int(np.mean(ws))}×{int(np.mean(hs))} px")

# ── 6. Memory check ──────────────────────────────────────────
print("\n[6] Memory:")
import psutil
ram = psutil.virtual_memory()
print(f"    RAM: {ram.used/1e9:.1f} GB used / {ram.total/1e9:.1f} GB total")
try:
    gpu_info = tf.config.experimental.get_memory_info("GPU:0")
    print(f"    GPU memory: {gpu_info['current']/1e9:.2f} GB used")
except:
    print("    GPU memory info unavailable (normal before first model run)")

# ── 7. Quick sanity model test ───────────────────────────────
print("\n[7] Quick pipeline sanity check …")
try:
    from sklearn.model_selection import StratifiedShuffleSplit

    all_fp, all_lb = [], []
    cls_list = sorted(common)
    c2i = {c:i for i,c in enumerate(cls_list)}
    for cls in cls_list:
        d = os.path.join(TRAIN_DIR, cls)
        if not os.path.isdir(d): continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpg",".jpeg",".png")):
                all_fp.append(os.path.join(d,f)); all_lb.append(c2i[cls])

    all_lb = np.array(all_lb)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for tr, va in sss.split(all_fp, all_lb): pass

    # Load 1 batch and check for NaN
    AUTOTUNE = tf.data.AUTOTUNE
    sample_fp = [all_fp[i] for i in tr[:32]]
    sample_lb = [all_lb[i] for i in tr[:32]]
    ds = tf.data.Dataset.from_tensor_slices((sample_fp, sample_lb))
    def load(p, l):
        img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (224,224))
        return tf.cast(img, tf.float32), l
    ds = ds.map(load).batch(32).prefetch(AUTOTUNE)
    for imgs, lbs in ds:
        nan_count = tf.reduce_sum(tf.cast(tf.math.is_nan(imgs), tf.int32)).numpy()
        inf_count = tf.reduce_sum(tf.cast(tf.math.is_inf(imgs), tf.int32)).numpy()
        print(f"    Batch shape : {imgs.shape}")
        print(f"    Label range : {lbs.numpy().min()} – {lbs.numpy().max()}")
        print(f"    Pixel range : {imgs.numpy().min():.1f} – {imgs.numpy().max():.1f}")
        print(f"    NaN pixels  : {nan_count}  (should be 0)")
        print(f"    Inf pixels  : {inf_count}  (should be 0)")
        break
    print("    ✓ Pipeline looks healthy.")
except Exception as e:
    print(f"    ✗ Pipeline error: {e}")

print("\n" + "=" * 55)
print("RECOMMENDATION:")
total_per_class = total_train / len(train_cls)
if total_per_class < 80:
    print(f"  ⚠ Very small dataset: avg {total_per_class:.0f} images/class.")
    print("    Consider data augmentation + pretrained features only.")
elif len(classes_lt50) > 5:
    print(f"  ⚠ {len(classes_lt50)} classes have <50 samples — heavy augmentation needed.")
else:
    print("  ✓ Dataset size looks reasonable.")
print("=" * 55)

DIAGNOSTIC REPORT

[1] GPU:
    Found: ['/physical_device:GPU:0']
    TF version: 2.19.0

[2] Class sample counts (train):
    Total train images : 2342
    Total test images  : 236
    Train classes      : 28
    Test  classes      : 27
    Min class (train)  : 'Tomato two spotted spider mites leaf' = 2 images
    Max class (train)  : 'Corn leaf blight' = 180 images
    Classes with <50 train images: 3
      • Cherry leaf: 47
      • Tomato leaf mosaic virus: 44
      • Tomato two spotted spider mites leaf: 2

[3] Class alignment:
    Common classes     : 27
    Only in train      : {'Tomato two spotted spider mites leaf'}
    Only in test       : set()

[4] Scanning for corrupted files …
    ✓ No corrupted files found.

[5] Image size sample (first 50 train images):
    Width  range : 180 – 3888 px
    Height range : 258 – 5184 px
    Avg size     : 1135×1095 px

[6] Memory:
    RAM: 11.8 GB used / 13.6 GB total
    GPU memory: 1.56 GB used

[7] Quick pipeline sanity check …
    Batc

In [ ]:
# ================================================================
# PlantDoc V4 — Targeted fixes based on diagnostic results
#
# EXACT PROBLEMS FOUND & FIXED:
#   1. 'Tomato two spotted spider mites leaf' has only 2 images
#      → Excluded from training (ruins loss landscape for all classes)
#   2. Only 87 avg images/class → aggressive augmentation + oversampling
#   3. RAM already at 11.8/13.6 GB → lean pipeline, no caching, batch=16
#   4. Native images are 1135×1095px → resize to 260px (not 300/384)
#      EfficientNetV2S was pretrained at 260×260 — exact native size
#   5. Mixup was corrupting val_accuracy metric (V2 bug) — fixed
#   6. Use SMOTE-style oversampling for minority classes in tf.data
# ================================================================

import os, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedShuffleSplit

# ── GPU ──────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Mixed precision — 2× faster on T4, frees ~3 GB VRAM
tf.keras.mixed_precision.set_global_policy("mixed_float16")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Config ───────────────────────────────────────────────────
IMG_SIZE     = (260, 260)   # EfficientNetV2S exact pretrain size
BATCH        = 16           # safe for T4 + 11.8 GB RAM already used
MIN_SAMPLES  = 10           # drop any class with fewer samples than this
LABEL_SMOOTH = 0.1
EPOCHS_HEAD  = 20
EPOCHS_FT1   = 30
EPOCHS_FT2   = 20
AUTOTUNE     = tf.data.AUTOTUNE

TRAIN_DIR = "/content/data/PlantDoc/train"
TEST_DIR  = "/content/data/PlantDoc/test"

# ================================================================
# STEP 1 — Class alignment + drop near-empty classes
# ================================================================
print("=" * 60)
print("STEP 1: Class setup …")

train_cls = sorted([d for d in os.listdir(TRAIN_DIR)
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))])
test_cls  = sorted([d for d in os.listdir(TEST_DIR)
                    if os.path.isdir(os.path.join(TEST_DIR, d))])
common    = sorted(set(train_cls) & set(test_cls))

# Count train samples per class
def count_imgs(cls_dir):
    if not os.path.isdir(cls_dir): return 0
    return sum(1 for f in os.listdir(cls_dir)
               if f.lower().endswith((".jpg",".jpeg",".png")))

# Drop classes with too few samples
CLASSES = []
dropped = []
for cls in common:
    n = count_imgs(os.path.join(TRAIN_DIR, cls))
    if n >= MIN_SAMPLES:
        CLASSES.append(cls)
    else:
        dropped.append((cls, n))

if dropped:
    print(f"  ⚠ Dropped {len(dropped)} class(es) with <{MIN_SAMPLES} train images:")
    for cls, n in dropped:
        print(f"      • '{cls}': {n} image(s)")

NUM_CLS  = len(CLASSES)
cls2idx  = {c: i for i, c in enumerate(CLASSES)}
idx2cls  = {i: c for c, i in cls2idx.items()}
print(f"  Training on {NUM_CLS} classes ({len(common)-NUM_CLS} dropped)\n")

# ================================================================
# STEP 2 — Collect paths with oversampling for minority classes
# ================================================================
print("STEP 2: Collecting & oversampling …")

all_fp, all_lb = [], []
class_counts   = {}

for cls in CLASSES:
    d   = os.path.join(TRAIN_DIR, cls)
    idx = cls2idx[cls]
    fps = [os.path.join(d, f) for f in sorted(os.listdir(d))
           if f.lower().endswith((".jpg",".jpeg",".png"))]
    class_counts[cls] = len(fps)
    all_fp.extend(fps)
    all_lb.extend([idx] * len(fps))

all_fp = np.array(all_fp)
all_lb = np.array(all_lb)

# Oversample minority classes to at least 80 samples
TARGET_MIN = 80
fp_aug, lb_aug = list(all_fp), list(all_lb)
rng = np.random.default_rng(SEED)

for cls in CLASSES:
    idx = cls2idx[cls]
    n   = class_counts[cls]
    if n < TARGET_MIN:
        needed  = TARGET_MIN - n
        cls_fps = all_fp[all_lb == idx]
        extra   = rng.choice(cls_fps, size=needed, replace=True)
        fp_aug.extend(extra)
        lb_aug.extend([idx] * needed)
        print(f"  Oversampled '{cls}': {n} → {n + needed}")

all_fp = np.array(fp_aug)
all_lb = np.array(lb_aug)

# Stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for tr_idx, va_idx in sss.split(all_fp, all_lb):
    pass
print(f"\n  Train: {len(tr_idx)} | Val: {len(va_idx)}")

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLS),
                           y=all_lb[tr_idx])
cw_dict = dict(enumerate(cw))

# ================================================================
# STEP 3 — Data pipelines
#   Training  : strong augmentation + Mixup (soft one-hot labels)
#   Validation : NO augmentation, HARD integer labels (critical fix)
# ================================================================
print("\nSTEP 3: Building pipelines …")

def load_img(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label

def augment(img, label):
    # Spatial
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))

    # Random crop (simulate zoom in/out)
    frac  = tf.random.uniform([], 0.80, 1.0)
    h     = tf.cast(tf.cast(IMG_SIZE[0], tf.float32) * frac, tf.int32)
    w     = tf.cast(tf.cast(IMG_SIZE[1], tf.float32) * frac, tf.int32)
    img   = tf.image.random_crop(img, [h, w, 3])
    img   = tf.image.resize(img, IMG_SIZE)

    # Colour jitter
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.08)

    # Gaussian noise
    img = img + tf.random.normal(tf.shape(img), stddev=6.0)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def cutout(img, label):
    """Random square erasing — helps with occlusion robustness."""
    if tf.random.uniform([]) > 0.5:
        cy  = tf.random.uniform([], IMG_SIZE[0]//5, 4*IMG_SIZE[0]//5, dtype=tf.int32)
        cx  = tf.random.uniform([], IMG_SIZE[1]//5, 4*IMG_SIZE[1]//5, dtype=tf.int32)
        sz  = IMG_SIZE[0] // 4
        y1  = tf.maximum(0, cy - sz//2); y2 = tf.minimum(IMG_SIZE[0], cy + sz//2)
        x1  = tf.maximum(0, cx - sz//2); x2 = tf.minimum(IMG_SIZE[1], cx + sz//2)
        pad = [[y1, IMG_SIZE[0]-y2], [x1, IMG_SIZE[1]-x2], [0, 0]]
        mask = tf.pad(tf.zeros([y2-y1, x2-x1, 3]), pad, constant_values=1.0)
        img  = img * mask
    return img, label

def mixup_batch(imgs, labels_oh, alpha=0.2):
    lam   = tf.constant(float(np.random.beta(alpha, alpha)), dtype=tf.float32)
    idx   = tf.random.shuffle(tf.range(tf.shape(imgs)[0]))
    imgs2 = tf.gather(imgs, idx)
    lbs2  = tf.gather(labels_oh, idx)
    return lam * imgs + (1-lam) * imgs2, lam * labels_oh + (1-lam) * lbs2

def make_train_ds(indices):
    fp = all_fp[indices]; lb = all_lb[indices]
    ds = tf.data.Dataset.from_tensor_slices((fp, lb))
    ds = ds.shuffle(len(fp), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_img,  num_parallel_calls=AUTOTUNE)
    ds = ds.map(augment,   num_parallel_calls=AUTOTUNE)
    ds = ds.map(cutout,    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    # ONE-HOT + MIXUP only on training
    ds = ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLS)),
                num_parallel_calls=AUTOTUNE)
    ds = ds.map(mixup_batch, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def make_val_ds(indices):
    """Hard integer labels → val_accuracy is real, not corrupted."""
    fp = all_fp[indices]; lb = all_lb[indices]
    ds = tf.data.Dataset.from_tensor_slices((fp, lb))
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

def make_test_ds():
    fps, lbs = [], []
    for cls in CLASSES:
        d = os.path.join(TEST_DIR, cls)
        if not os.path.isdir(d): continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpg",".jpeg",".png")):
                fps.append(os.path.join(d, f))
                lbs.append(cls2idx[cls])
    ds = tf.data.Dataset.from_tensor_slices((fps, lbs))
    ds = ds.map(load_img, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE), fps, lbs

train_ds              = make_train_ds(tr_idx)
val_ds                = make_val_ds(va_idx)
test_ds, t_fps, t_lbs = make_test_ds()
print(f"  Test samples: {len(t_lbs)}\n")

# ================================================================
# STEP 4 — Model: EfficientNetV2S @ 260×260
# ================================================================
print("STEP 4: Building model …")

base = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(260, 260, 3),
    include_preprocessing=True   # no external preprocess_input needed
)
base.trainable = False

def build_model():
    inp = tf.keras.Input(shape=(260, 260, 3))
    x   = base(inp, training=False)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)
    x   = tf.keras.layers.BatchNormalization()(x)
    # Wide head — compensates for small dataset
    x   = tf.keras.layers.Dense(
              1024,
              kernel_regularizer=tf.keras.regularizers.l2(1e-4),
              kernel_initializer="he_normal")(x)
    x   = tf.keras.layers.Activation("relu")(x)
    x   = tf.keras.layers.Dropout(0.5)(x)
    x   = tf.keras.layers.Dense(
              512,
              kernel_regularizer=tf.keras.regularizers.l2(1e-4),
              kernel_initializer="he_normal")(x)
    x   = tf.keras.layers.Activation("relu")(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    # float32 output required with mixed_float16 policy
    out = tf.keras.layers.Dense(NUM_CLS, activation="softmax",
                                 dtype="float32")(x)
    return tf.keras.Model(inp, out)

model = build_model()

# ================================================================
# STEP 5 — Custom loss (handles both soft + hard labels)
# ================================================================
class SmartCE(tf.keras.losses.Loss):
    def __init__(self, smoothing=LABEL_SMOOTH, n=NUM_CLS, **kw):
        super().__init__(**kw)
        self.smoothing = smoothing; self.n = n

    def call(self, y_true, y_pred):
        y_pred = tf.cast(tf.clip_by_value(y_pred, 1e-7, 1.0), tf.float32)
        if y_true.shape.rank == 1 or (y_true.shape.rank==2 and y_true.shape[-1]==1):
            y_true = tf.one_hot(tf.cast(tf.squeeze(y_true), tf.int32), self.n)
        y_true = tf.cast(y_true, tf.float32)
        smooth = self.smoothing / tf.cast(self.n, tf.float32)
        y_true = y_true * (1.0 - self.smoothing) + smooth
        return -tf.reduce_mean(tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1))

    def get_config(self):
        return {"smoothing": self.smoothing, "n": self.n}

loss_fn = SmartCE()

# ── LR schedule with linear warmup ───────────────────────────
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, lr_max, total_steps, warmup_steps):
        super().__init__()
        self.lr_max = lr_max
        self.total  = float(total_steps)
        self.warmup = float(warmup_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = self.lr_max * (step / self.warmup)
        cos    = 0.5 * self.lr_max * (
                     1.0 + tf.cos(np.pi * (step - self.warmup) /
                                   (self.total - self.warmup)))
        return tf.where(step < self.warmup, warmup, cos)

    def get_config(self):
        return {"lr_max": self.lr_max, "total_steps": self.total,
                "warmup_steps": self.warmup}

steps = len(tr_idx) // BATCH + 1

# ================================================================
# STEP 6 — Callbacks
# ================================================================
def callbacks(ckpt, patience_es=7, patience_lr=3):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=patience_es,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, monitor="val_accuracy",
            save_best_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", factor=0.4,
            patience=patience_lr, min_lr=1e-8, verbose=1),
    ]

# ================================================================
# STEP 7 — Phase 1: Head only (frozen backbone)
# ================================================================
print("STEP 5: Phase 1 — head only …")

sched1 = WarmupCosine(lr_max=1e-3,
                       total_steps=EPOCHS_HEAD * steps,
                       warmup_steps=3 * steps)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=sched1, weight_decay=1e-4, clipnorm=1.0),
    loss=loss_fn,
    metrics=["accuracy"]
)

h1 = model.fit(train_ds, validation_data=val_ds,
               epochs=EPOCHS_HEAD,
               class_weight=cw_dict,
               callbacks=callbacks("ckpt1.keras"))

best_val_h1 = max(h1.history["val_accuracy"])
print(f"\n  Best val_accuracy Phase 1: {best_val_h1*100:.2f}%")

# ================================================================
# STEP 8 — Phase 2: Unfreeze top 60 layers
# ================================================================
print("\nSTEP 6: Phase 2 — unfreeze top 60 layers …")

base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
# ALWAYS keep BatchNorm frozen — critical for small datasets
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

sched2 = WarmupCosine(lr_max=5e-5,
                       total_steps=EPOCHS_FT1 * steps,
                       warmup_steps=2 * steps)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=sched2, weight_decay=1e-5, clipnorm=1.0),
    loss=loss_fn,
    metrics=["accuracy"]
)

h2 = model.fit(train_ds, validation_data=val_ds,
               epochs=EPOCHS_FT1,
               class_weight=cw_dict,
               callbacks=callbacks("ckpt2.keras", patience_es=8))

best_val_h2 = max(h2.history["val_accuracy"])
print(f"\n  Best val_accuracy Phase 2: {best_val_h2*100:.2f}%")

# ================================================================
# STEP 9 — Phase 3: Full backbone, very conservative LR
# ================================================================
print("\nSTEP 7: Phase 3 — full backbone …")

for layer in base.layers:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

sched3 = WarmupCosine(lr_max=1e-5,
                       total_steps=EPOCHS_FT2 * steps,
                       warmup_steps=2 * steps)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=sched3, weight_decay=1e-5, clipnorm=0.5),
    loss=loss_fn,
    metrics=["accuracy"]
)

h3 = model.fit(train_ds, validation_data=val_ds,
               epochs=EPOCHS_FT2,
               class_weight=cw_dict,
               callbacks=callbacks("best_v4.keras", patience_es=8, patience_lr=4))

best_val_h3 = max(h3.history["val_accuracy"])
print(f"\n  Best val_accuracy Phase 3: {best_val_h3*100:.2f}%")

# ================================================================
# STEP 10 — Load best checkpoint
# ================================================================
print("\nSTEP 8: Loading best checkpoint …")
model = tf.keras.models.load_model(
    "best_v4.keras",
    custom_objects={"SmartCE": SmartCE, "WarmupCosine": WarmupCosine}
)

# ================================================================
# STEP 11 — Deterministic TTA (8 passes: 4 rotations × flip)
# ================================================================
print("  Running TTA inference …")

def tta_predict(model, dataset):
    all_probs = []
    all_true  = []
    for imgs, lbs in dataset:
        batch_preds = []
        for k in range(4):
            rot  = tf.image.rot90(imgs, k)
            p1   = model(rot,                         training=False).numpy()
            p2   = model(tf.image.flip_left_right(rot), training=False).numpy()
            batch_preds.extend([p1, p2])
        all_probs.append(np.mean(batch_preds, axis=0))
        all_true.extend(lbs.numpy())
    return np.concatenate(all_probs, axis=0), np.array(all_true)

tta_probs, y_true = tta_predict(model, test_ds)
y_pred_tta        = np.argmax(tta_probs, axis=1)

# Standard (no TTA) for comparison
std_probs = []
for imgs, _ in test_ds:
    std_probs.append(model(imgs, training=False).numpy())
std_probs  = np.concatenate(std_probs, axis=0)
y_pred_std = np.argmax(std_probs, axis=1)

# ================================================================
# STEP 12 — Full metrics
# ================================================================
def metrics(y_t, y_p, label):
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(f"  Accuracy           : {accuracy_score(y_t,y_p)*100:.2f}%")
    print(f"  Weighted Precision : {precision_score(y_t,y_p,average='weighted',zero_division=0):.4f}")
    print(f"  Weighted Recall    : {recall_score(y_t,y_p,average='weighted',zero_division=0):.4f}")
    print(f"  Weighted F1        : {f1_score(y_t,y_p,average='weighted',zero_division=0):.4f}")
    print(f"  Macro F1           : {f1_score(y_t,y_p,average='macro',zero_division=0):.4f}")

print("\n" + "=" * 55)
print("     EVALUATION RESULTS")
print("=" * 55)
metrics(y_true, y_pred_std, "Standard Inference")
metrics(y_true, y_pred_tta, "TTA Inference (8 passes)")
print("=" * 55)

print("\nPer-class report (TTA):")
print(classification_report(y_true, y_pred_tta,
                             target_names=CLASSES, zero_division=0))

# ================================================================
# STEP 13 — Training curves
# ================================================================
def cat(h, k): return h.history.get(k, [])
def c3(k):     return cat(h1,k) + cat(h2,k) + cat(h3,k)

acc_tr  = c3("accuracy");    acc_val  = c3("val_accuracy")
loss_tr = c3("loss");        loss_val = c3("val_loss")
ep      = range(1, len(acc_tr)+1)
pb1     = len(cat(h1,"accuracy"))
pb2     = pb1 + len(cat(h2,"accuracy"))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"PlantDoc V4 — Training History  "
             f"(Best val acc: {max(acc_val)*100:.1f}%)",
             fontsize=14, fontweight="bold")

spans  = [(1, pb1, "#1565C0", "Phase 1\nHead"),
          (pb1, pb2, "#2E7D32", "Phase 2\nTop-60"),
          (pb2, len(ep), "#E65100", "Phase 3\nFull")]

for ax, (tr_d, val_d, title) in zip(
    axes,
    [(acc_tr,  acc_val,  "Accuracy"),
     (loss_tr, loss_val, "Loss")]
):
    ax.plot(ep, tr_d,  color="#1565C0", lw=2, label="Train")
    ax.plot(ep, val_d, color="#B71C1C", lw=2, ls="--", label="Val", alpha=0.85)
    for s, e_, col, lbl in spans:
        ax.axvspan(s, e_, alpha=0.07, color=col)
        if s > 1:
            ax.axvline(s, color=col, lw=1.2, ls=":", label=lbl)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_v4.png", dpi=150, bbox_inches="tight")
print("\nSaved: training_curves_v4.png")

# ================================================================
# STEP 14 — Confusion matrices (raw + normalised)
# ================================================================
cm      = confusion_matrix(y_true, y_pred_tta)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
snames  = [c[:22] for c in CLASSES]

fig, axes = plt.subplots(1, 2, figsize=(30, 22))
fig.suptitle(f"Confusion Matrix — PlantDoc V4 (TTA)  "
             f"|  Accuracy: {accuracy_score(y_true,y_pred_tta)*100:.1f}%",
             fontsize=15, fontweight="bold")

for ax, data, title, vmax in zip(
    axes,
    [cm,      cm_norm],
    ["Counts","Normalised (row %)"],
    [None,    1.0]
):
    im = ax.imshow(data, cmap="Blues", vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.025)
    ax.set_xticks(range(NUM_CLS)); ax.set_yticks(range(NUM_CLS))
    ax.set_xticklabels(snames, rotation=65, ha="right", fontsize=7)
    ax.set_yticklabels(snames, fontsize=7)
    thresh = data.max() / 2.0
    for i in range(NUM_CLS):
        for j in range(NUM_CLS):
            if cm[i, j] > 0:
                txt = str(cm[i,j]) if title=="Counts" else f"{cm_norm[i,j]:.2f}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=5.5,
                        color="white" if data[i,j] > thresh else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)

plt.tight_layout()
plt.savefig("confusion_matrix_v4.png", dpi=150, bbox_inches="tight")
print("Saved: confusion_matrix_v4.png")

# ================================================================
# STEP 15 — Per-class precision / recall / F1 bar chart
# ================================================================
pf1  = f1_score(y_true, y_pred_tta, average=None, zero_division=0)
ppre = precision_score(y_true, y_pred_tta, average=None, zero_division=0)
prec = recall_score(y_true, y_pred_tta, average=None, zero_division=0)
sidx = np.argsort(pf1)
x    = np.arange(NUM_CLS); w = 0.27

fig, ax = plt.subplots(figsize=(13, 13))
ax.barh(x-w,  pf1[sidx],  w, label="F1",       color="#1565C0", edgecolor="white", lw=0.4)
ax.barh(x,    ppre[sidx], w, label="Precision", color="#2E7D32", edgecolor="white", lw=0.4)
ax.barh(x+w,  prec[sidx], w, label="Recall",    color="#B71C1C", edgecolor="white", lw=0.4)
ax.set_yticks(x)
ax.set_yticklabels([snames[i] for i in sidx], fontsize=8)
ax.axvline(0.5, color="gray",  ls="--", alpha=0.5, lw=1)
ax.axvline(0.7, color="green", ls="--", alpha=0.5, lw=1)
ax.set_xlim(0, 1.18); ax.set_xlabel("Score")
ax.set_title("Per-class Precision / Recall / F1  (sorted by F1)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="x", alpha=0.3)
for data_arr, offset in [(pf1[sidx],-w), (ppre[sidx],0), (prec[sidx],w)]:
    for j, val in enumerate(data_arr):
        ax.text(val+0.01, x[j]+offset+w/2, f"{val:.2f}", va="center", fontsize=6.5)
plt.tight_layout()
plt.savefig("per_class_metrics_v4.png", dpi=150, bbox_inches="tight")
print("Saved: per_class_metrics_v4.png")

# ── Top confused pairs ────────────────────────────────────────
print("\nTop 8 confused pairs (TTA):")
cm2 = cm.copy(); np.fill_diagonal(cm2, 0)
for rank, fi in enumerate(np.argsort(cm2.ravel())[::-1][:8], 1):
    tc = idx2cls[fi // NUM_CLS]; pc = idx2cls[fi % NUM_CLS]
    n  = cm2.ravel()[fi]
    if n > 0:
        print(f"  {rank}. {tc:<33s} → {pc:<33s} ({n}×)")

print(f"\n✅ Done!  Best val_accuracy = {max(acc_val)*100:.1f}%")
print("   Outputs: training_curves_v4.png | confusion_matrix_v4.png | per_class_metrics_v4.png")

STEP 1: Class setup …


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/PlantDoc/train'